# Bloc 1 — Python pour la Data Science (B2)
## Séance 0 + Séance 1 — Installation + premières illusions sur des données Google Trends

**Decision problem:** how can a public signal become evidence for a product or business decision without overclaiming?

Ce notebook contient **deux parties** :
- **Séance 0 (onboarding)** : installation + vérifications + “ça tourne”
- **Séance 1 (3h30)** : introduction Data Science + NumPy + premières explorations **sur Google Trends récupéré automatiquement**

> ⚠️ Google Trends n’a pas d’API officielle. On utilise **pytrends** (non-officiel).  
> Si Google limite temporairement les requêtes (429 / captcha), réessayez plus tard ou changez de réseau.


# Séance 0 — Installation & prise en main (objectif : zéro friction en séance 1)

## Objectif
À la fin de cette séance, vous devez pouvoir :
- exécuter un notebook
- installer/charger les bibliothèques
- afficher un graphique
- récupérer des données Google Trends via `pytrends`

## Où exécuter ce notebook ?
### Option A (recommandée) — Anaconda / Jupyter
1. Installer Anaconda
2. Lancer **JupyterLab** (ou Jupyter Notebook)
3. Ouvrir ce notebook
4. Exécuter les cellules **dans l’ordre** (Shift+Enter)

### Option B — VS Code
1. Installer Python 3.10+
2. Installer VS Code + extensions Python & Jupyter
3. Ouvrir ce notebook
4. Exécuter les cellules **dans l’ordre**

### Option C — Google Colab (si votre machine pose problème)
1. Ouvrir Colab
2. Importer ce notebook
3. Exécuter les cellules (l’installation sera refaite à chaque session)


## 0.1 — Vérifier Python

In [ ]:
import sys, platform
print("Python:", sys.version)
print("Platform:", platform.platform())

## 0.2 — Installer les bibliothèques (si nécessaire)

- Si vous êtes sur **Anaconda**, vous avez souvent déjà `numpy/pandas/matplotlib/seaborn/sklearn`.
- Sinon, exécutez la cellule ci-dessous.
- On installe aussi **pytrends** pour récupérer Google Trends.


In [ ]:
# Installe (ou met à jour) les dépendances nécessaires
# Note: sur certains environnements, il faut relancer le kernel après installation.
!pip -q install --upgrade numpy pandas matplotlib seaborn scikit-learn pytrends

## 0.3 — Tester les imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

from pytrends.request import TrendReq

print("✅ Imports OK")
print("numpy", np.__version__)
print("pandas", pd.__version__)
print("sklearn", sklearn.__version__)

## 0.4 — Test rapide : calcul + graphique

In [ ]:
data = np.array([1, 2, 3, 4, 6, 9, 15])
print("mean =", data.mean(), "| std =", data.std())

plt.figure(figsize=(10, 3))
plt.plot(data, marker="o")
plt.title("Test graphique (doit s'afficher)")
plt.show()

## 0.5 — Test Google Trends (connexion + première requête)

On récupère une série d’intérêt dans le temps pour un mot-clé.
- `geo="FR"` : France
- `timeframe="today 12-m"` : 12 derniers mois


“Les warnings des bibliothèques externes peuvent être ignorés si le résultat est correct.”

In [ ]:
# ⚠️ pytrends est non-officiel : Google peut limiter les requêtes.
# Si vous obtenez une erreur 429/captcha, réessayez plus tard ou changez de réseau.

kw = "chatgpt"   # changez si vous voulez
geo = "FR"
timeframe = "today 12-m"

pytrends = TrendReq(hl="fr-FR", tz=360)  # tz en minutes (360 = UTC+6, peu critique ici)
pytrends.build_payload([kw], cat=0, timeframe=timeframe, geo=geo, gprop="")

iot = pytrends.interest_over_time()
iot.head()

In [ ]:
# Visualisation rapide
if not iot.empty:
    plt.figure(figsize=(12, 4))
    plt.plot(iot.index, iot[kw], label=kw)
    plt.title(f"Google Trends — {kw} ({geo}, {timeframe})")
    plt.xlabel("Date")
    plt.ylabel("Intérêt (0–100)")
    plt.legend()
    plt.show()
else:
    print("Aucune donnée retournée (essayez un autre mot-clé ou timeframe).")

✅ Si vous voyez une courbe, vous êtes prêt pour la Séance 1.

---

# Séance 1 — Introduction à la Data Science avec Python (3h30)
**Syllabus :**
- Panorama Data Science et applications
- Environnement Python DS (déjà fait en séance 0)
- Manipulation de données avec **NumPy**


## 1.1 — Qu’est-ce qu’un “problème Data Science” ? (version courte)
On part d’une question, puis on vérifie ce que **les données permettent** (ou non) de conclure.

Fil rouge du semestre :
> **Détecter des signaux faibles** à partir de données Google Trends.


## 1.2 — NumPy : mesurer variation, vitesse, accélération (sur un mini-signal)
Ce sont des briques simples, mais très puissantes pour les séries temporelles.


In [ ]:
signal = np.array([5, 6, 6, 7, 8, 10, 13, 18, 22, 30])
signal

In [ ]:
print("mean =", signal.mean(), "| std =", signal.std())

In [ ]:
variation = np.diff(signal)           # vitesse
acceleration = np.diff(variation)      # accélération
variation, acceleration

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(signal, label="signal")
plt.plot(range(1, len(signal)), variation, label="vitesse (diff)")
plt.legend()
plt.title("Signal vs vitesse")
plt.show()

## 1.3 — Récupérer des données Google Trends (automatique)

On récupère **plusieurs mots-clés** et on observe les premières illusions :
- un score est **relatif** (0–100) sur une fenêtre
- comparer deux courbes “brutes” est souvent trompeur


In [ ]:
# Paramètres de la séance (vous pouvez changer)
kw_list = ["chatgpt", "bitcoin", "taylor swift"]  # exemples
geo = "FR"
timeframe = "today 5-y"  # 5 dernières années (bien pour illustrer)


In [ ]:
print("kw_list =", kw_list)
print("geo =", geo)
print("timeframe =", timeframe)

In [ ]:
from pytrends.request import TrendReq
from pytrends.exceptions import TooManyRequestsError
from pathlib import Path
import pandas as pd

Path("data/snapshots").mkdir(parents=True, exist_ok=True)

def _safe_kw_sig(kw_list, max_len=60):
    # Normalisation + tri pour que ["a","b"] == ["b","a"]
    norm = [k.strip().lower().replace(" ", "_") for k in kw_list]
    norm = sorted(norm)
    safe = "_".join(norm)
    return safe[:max_len]

def save_snapshot_csv_kwlist(kw_list, geo, timeframe, filename=None):
    pt = TrendReq(hl="fr-FR", tz=360)
    pt.build_payload(kw_list, timeframe=timeframe, geo=geo)

    try:
        iot = pt.interest_over_time()
    except TooManyRequestsError:
        print("⚠️ Google bloque (429). Réessaie plus tard / change de réseau.")
        return None

    if iot is None or iot.empty:
        print("⚠️ Aucune donnée retournée.")
        return None

    if "isPartial" in iot.columns:
        iot = iot.drop(columns=["isPartial"])

    if filename is None:
        sig = _safe_kw_sig(kw_list)
        filename = f"iot_{geo}_{timeframe.replace(' ', '_')}_{sig}.csv"

    out = Path("data/snapshots") / filename
    iot.to_csv(out, index=True)
    print("✅ Saved:", out, "| rows=", len(iot), "| cols=", list(iot.columns))
    return out

kw_list = ["chatgpt", "iphone", "meteo"]
p1 = save_snapshot_csv_kwlist(kw_list, "FR", "today 12-m")
p2 = save_snapshot_csv_kwlist(kw_list, "FR", "today 5-y")

# test relecture (si snapshot créé)
if p1:
    df = pd.read_csv(p1, index_col=0, parse_dates=True)
    df.head()

In [ ]:
import time
from pathlib import Path
import pandas as pd

from pytrends.request import TrendReq
from pytrends.exceptions import TooManyRequestsError


def fetch_trends_iot(
    kw_list,
    geo="FR",
    timeframe="today 12-m",
    hl="fr-FR",
    tz=360,
    retries=5,
    base_sleep=5,
):
    pytrends = TrendReq(hl=hl, tz=tz)
    pytrends.build_payload(kw_list, cat=0, timeframe=timeframe, geo=geo, gprop="")

    for attempt in range(1, retries + 1):
        try:
            iot = pytrends.interest_over_time()
            return iot
        except TooManyRequestsError:
            sleep_s = base_sleep * (2 ** (attempt - 1))
            print(f"⚠️ 429 Too Many Requests. Retry {attempt}/{retries} in {sleep_s}s...")
            time.sleep(sleep_s)

    raise RuntimeError("Toujours bloqué par Google (429). Réessayez plus tard / changez de réseau.")


def cache_path_for(kw_list, geo, timeframe):
    safe = "_".join([k.replace(" ", "_") for k in kw_list])
    safe = safe[:80]
    tf = timeframe.replace(" ", "_").replace("/", "_")
    return Path("data") / f"iot_{geo}_{tf}_{safe}.csv"


def get_iot_with_cache(kw_list, geo="FR", timeframe="today 12-m"):
    Path("data").mkdir(exist_ok=True)
    cache_path = cache_path_for(kw_list, geo, timeframe)

    try:
        iot = fetch_trends_iot(kw_list, geo=geo, timeframe=timeframe)

        # Nettoyage léger : retirer isPartial si présent
        if "isPartial" in iot.columns:
            iot = iot.drop(columns=["isPartial"])

        iot.to_csv(cache_path, index=True)
        print(f"✅ Données récupérées et mises en cache: {cache_path}")
        return iot

    except Exception as e:
        if cache_path.exists():
            print(f"⚠️ Live bloqué ({type(e).__name__}). Utilisation du cache: {cache_path}")
            return pd.read_csv(cache_path, index_col=0, parse_dates=True)
        raise

In [ ]:
from pathlib import Path
import pandas as pd

def snapshot_path(geo, timeframe):
    tf = timeframe.replace(" ", "_").replace("/", "_")
    return Path("data/snapshots") / f"iot_{geo}_{tf}.csv"

def load_iot_live_or_snapshot(kw_list, geo="FR", timeframe="today 5-y"):
    snap = snapshot_path(geo, timeframe)
    snap.parent.mkdir(parents=True, exist_ok=True)

    try:
        from pytrends.request import TrendReq
        pt = TrendReq(hl="fr-FR", tz=360)
        pt.build_payload(kw_list, timeframe=timeframe, geo=geo)
        iot = pt.interest_over_time()
        if iot is None or iot.empty:
            raise RuntimeError("No data returned")

        # Nettoyage léger : retirer isPartial si présent
        if "isPartial" in iot.columns:
            iot = iot.drop(columns=["isPartial"])

        print("✅ LIVE Google Trends OK")
        # optionnel: on met en snapshot quand ça marche
        iot.to_csv(snap, index=True)
        return iot

    except Exception as e:
        if snap.exists():
            print(f"⚠️ LIVE indisponible ({type(e).__name__}). Snapshot: {snap}")
            return pd.read_csv(snap, index_col=0, parse_dates=True)

        raise RuntimeError(
            f"Google Trends indisponible et snapshot absent ({snap}). "
            f"Réessayez plus tard ou ajoutez un snapshot au repo."
        ) from e

In [ ]:
# Nettoyage minimal : retirer la colonne isPartial si présente
df = iot.copy()
if "isPartial" in df.columns:
    df = df.drop(columns=["isPartial"])

df.tail()

In [ ]:
geo = "FR"
kw = "chatgpt"  # mot-clé fixe pour la démo "fenêtre temporelle"

iot_12m = load_iot_live_or_snapshot([kw], geo=geo, timeframe="today 12-m")
iot_5y  = load_iot_live_or_snapshot([kw], geo=geo, timeframe="today 5-y")

def clean_iot(iot, kw):
    df = iot.copy()
    if "isPartial" in df.columns:
        df = df.drop(columns=["isPartial"])
    # si la colonne kw n’existe pas (réponse partielle), on prend la 1ère colonne dispo
    if kw not in df.columns:
        available = [c for c in df.columns if c != "isPartial"]
        kw_used = available[0]
        print(f"⚠️ '{kw}' absent. Utilisation de '{kw_used}' à la place.")
        kw = kw_used
    return df[[kw]]

a12 = clean_iot(iot_12m, kw).rename(columns={kw: f"{kw} (12m)"})
a5y = clean_iot(iot_5y,  kw).rename(columns={kw: f"{kw} (5y)"})

plt.figure(figsize=(12, 5))
plt.plot(a12.index, a12.iloc[:, 0], label=a12.columns[0])
plt.plot(a5y.index, a5y.iloc[:, 0], label=a5y.columns[0], alpha=0.8)
plt.title(f"Même mot-clé, fenêtres différentes ({geo})")
plt.ylabel("Intérêt (0–100)")
plt.legend()
plt.show()

## 1.4 — Première illusion : le score (0–100) n’est pas absolu

On vient de tracer **le même mot-clé** sur deux fenêtres temporelles différentes (12 mois vs 5 ans).

Questions (à discuter) :
- Que vaut un **100** sur Google Trends ?
- Est-ce qu’un **70** aujourd’hui est comparable à un **70** il y a 3 ans ?
- Que change le choix de la **fenêtre temporelle** sur l’interprétation ?

👉 Conclusion : le score est **relatif** à la fenêtre et au maximum observé.


## 1.5 — Un premier indicateur simple : lissage (moyenne glissante)
Même un lissage **change** ce qu’on croit voir.


In [ ]:
# Colonnes réellement disponibles
cols = [c for c in df.columns if c != "isPartial"]

# Diagnostic pédagogique
missing = [kw for kw in kw_list if kw not in cols]
if missing:
    print("⚠️ Mots-clés manquants (réponse partielle / cache) :", missing)
print("✅ Colonnes tracées :", cols)

window = 8
df_smooth = df[cols].rolling(window=window, min_periods=1).mean()

plt.figure(figsize=(12, 5))
for kw in cols:
    plt.plot(df.index, df[kw], alpha=0.25, label=f"{kw} (brut)")
    plt.plot(df_smooth.index, df_smooth[kw], label=f"{kw} (lissé)")
plt.title(f"Brut vs lissé (window={window})")
plt.xlabel("Date")
plt.ylabel("Intérêt (0–100)")
plt.legend(ncols=2)
plt.show()

## 1.6 — Mini-exercice (pendant la séance)

1) Choisissez 3 mots-clés (au choix : sport, musique, tech, finance…).  
2) Récupérez 5 ans de données en France.  
3) Tracez le brut et le lissé.  
4) Notez **3 problèmes** qui empêchent une conclusion solide (comparabilité, fenêtre, bruit, etc.).

> C’est volontaire : en séance 2, on rendra ces données **comparables**.


## 1.7 — Conclusion (séance 1)
- Nous avons manipulé des données numériques avec NumPy
- Nous avons récupéré des données réelles (Google Trends) automatiquement
- Nous avons constaté que les données brutes sont **trompeuses** et **non directement comparables**

➡️ Séance 2 : Pandas + nettoyage + alignement + rendre comparables plusieurs requêtes.
